# YOLO Hyperparameters Explained ⚙️
When you run `yolo train`, there are dozens of settings (hyperparameters) you can tweak. Getting these right is the difference between a model that trains in 1 hour with 90% accuracy, and a model that trains in 10 hours with 50% accuracy.

This notebook explains the most important hyperparameters you can configure.

## 1. The Core Settings
These are the settings you will change on almost every single training run.

### **`epochs`** (Default: 100)
* **What it is:** How many times the model will see your *entire* dataset.
* **When to change it:** If your model's loss is still going down rapidly at the end of training, you need more epochs (e.g., `epochs=300`). If your dataset is massive (e.g., 1 million images), you need fewer epochs (e.g., `epochs=50`).

### **`batch`** (Default: 16)
* **What it is:** How many images the model looks at simultaneously before it updates its brain. 
* **How to tune it:** You want this as **high as possible** without crashing your GPU (`CUDA Out of Memory` error). Higher batches mean faster training and smoother learning. Try `batch=32` or `batch=64` if your GPU can handle it. If it crashes, drop it to `batch=8`.

### **`imgsz`** (Default: 640)
* **What it is:** The size that all images are resized to before entering the model (e.g., 640x640 pixels).
* **How to tune it:** If you are trying to detect tiny objects (like a tiny screw on a massive pipeline), increase this to `imgsz=1024` or `1280` so the model can actually see the details. **Warning:** Larger images require exponentially more GPU memory, so you will have to lower your `batch` size.

## 2. Preventing Overfitting
These settings stop the model from memorizing your data.

### **`patience`** (Default: 50)
* **What it is:** "Early Stopping". If the model's validation accuracy doesn't improve for 50 epochs in a row, YOLO will automatically kill the training early.
* **Why it's amazing:** You can safely set `epochs=1000` and `patience=50`. You can go to sleep knowing YOLO will stop training the exact moment the model stops getting better, saving you hours of wasted GPU time.

### **`dropout`** (Default: 0.0)
* **What it is:** Randomly "turns off" a percentage of the model's brain cells during training. 
* **Why use it:** If your model is memorizing the training data (overfitting), setting `dropout=0.1` or `0.2` forces the model to learn multiple different ways to recognize an object, making it much more robust in the real world.

## 3. The Brain Tweaks (Optimization)
These settings control how the math works under the hood.

### **`optimizer`** (Default: 'auto')
* **What it is:** The algorithm that updates the weights. 'auto' usually picks AdamW for small datasets and SGD for large ones. 
* **When to change it:** If you have a huge dataset, forcing `optimizer='SGD'` is often better. If you have a tiny dataset, forcing `optimizer='AdamW'` learns much faster.

### **`lr0`** (Initial Learning Rate) and **`lrf`** (Final Learning Rate)
* **What it is:** The size of the "steps" the model takes to reach the answer.
* **How it works:** Training starts with large steps (`lr0`, usually around 0.01) to get close to the answer quickly. As training finishes, it uses tiny steps (`lrf`, usually 0.01 * lr0) to perfectly pinpoint the optimal answer without overshooting it.

### **`weight_decay`** (Default: 0.0005)
* **What it is:** A mathematical penalty that prevents any single "brain cell" in the model from becoming too powerful or important. It keeps the model balanced and prevents overfitting.

## 4. Data Augmentations (Making fake data)
YOLO manipulates your training images on-the-fly to create infinite variations. This is how it learns to recognize a valve even if it's upside down or in the dark.

### **`mosaic`** (Default: 1.0)
* **What it is:** Takes 4 different training images and smashes them together into one single image.
* **Why it's brilliant:** It forces the model to look at objects in totally weird, unexpected contexts, and trains it to detect objects at a smaller scale.

### **`degrees`** (Rotation) & **`flipud`** (Flip Up/Down)
* **What it is:** Randomly rotates the image or flips it upside down.
* **When to use it:** If you are detecting items on a factory conveyor belt, a part could be rotated at any angle. Set `degrees=180` and `flipud=0.5`. 
* **When to AVOID it:** If you are detecting cars on a road, cars are almost never upside down. Using `flipud` here would confuse the model and make it worse! Always think: *"Does this happen in the real world?"*

### **`hsv_h`, `hsv_s`, `hsv_v`** (Color, Saturation, Brightness)
* **What it is:** Randomly changes the lighting and colors of the image.
* **Why use it:** If you train your model on a sunny day, it will fail when it's cloudy. By changing the brightness (`hsv_v`), the model learns to ignore the lighting and focus on the actual shape of the object.

---
### 🚀 Example: The "Tiny Dataset" Command
If you only have 100 images, you want aggressive early stopping, high dropout, and heavy data augmentation to squeeze out every drop of learning:

```python
model.train(
    data='data.yaml',
    epochs=500,
    patience=50,       # Stop early if no improvement
    batch=16,
    optimizer='AdamW', # Great for small data
    dropout=0.2,       # Prevent memorization
    degrees=90,        # Rotate images
    hsv_v=0.4          # Change lighting heavily
)
```